# Stage1 发行人份额创设：条件性开发期信息检验

冻结原 Stage1 的39项配置输入和五个开盘区间SOXX标签，只增加“负20日趋势下、发行人份额五交易日净变化（滞后两会话）”。发行人当前历史表、三个旧版和已披露拆股单位已绑定。两会话延迟只是未证实的历史可用性假设；结果不授权经济账户或PIT声明。

In [1]:
import torch
import json,hashlib
from pathlib import Path
import numpy as np,pandas as pd,qlib,sklearn
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from src.data.qlib_native import build_qlib_store_from_frame
from src.data.soxx_issuer_fund_download import read_historical
from src.data.minute_intake import atomic_json
root=Path.cwd();card=root/'research/stage1_issuer_share_creation_information_20261022';plan=json.loads((card/'plan.json').read_text())
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert sklearn.__version__==plan['model']['runtime_scikit_learn']
spec=plan['candidate_source'];source_card=root/spec['card'];parent=plan['parent'];parent_card=root/parent['source_card']
for key,file in [('plan_sha256','plan.json'),('summary_sha256','summary.json'),('source_audit_sha256','source_audit.json'),('workbook_vintage_audit_sha256','workbook_vintage_audit.json'),('source_gate_review_sha256','source_gate_review.json')]:
 assert sha(source_card/file)==spec[key],('source changed',file)
assert sha(root/spec['current_issuer_workbook'])==spec['current_issuer_workbook_sha256']
assert sha(root/spec['parser'])==spec['parser_sha256']
for name,digest in spec['readable_archived_workbook_sha256'].items():assert sha(root/name)==digest
assert sha(root/spec['calendar_provider']/'qlib_store_manifest.json')==spec['calendar_manifest_sha256']
assert json.loads((source_card/'nas_source_archive_receipt.json').read_text())['archive_sha256']==spec['source_nas_archive_sha256']
assert json.loads((source_card/'nas_workbook_vintage_archive_receipt.json').read_text())['archive_sha256']==spec['workbook_vintage_nas_archive_sha256']
source_audit=json.loads((source_card/'source_audit.json').read_text())
vintage=json.loads((source_card/'workbook_vintage_audit.json').read_text())
review=json.loads((source_card/'source_gate_review.json').read_text())
assert source_audit['qlib_train_sessions']==3020 and vintage['all_calendar_share_mismatches_total']==0
assert review['verdict']=='ALLOW_SEPARATE_CONDITIONAL_RETROSPECTIVE_INFORMATION_SCREEN_ONLY'
for name,key in [('inputs/features','feature_artifact_sha256'),('inputs/daily_frame','frame_artifact_sha256'),('inputs/timing','timing_artifact_sha256'),('models/base_outer_71','base_model_artifact_sha256')]:
 assert sha(parent_card/'qlib_recorder'/parent['experiment_id']/parent['recorder_id']/'artifacts'/name)==parent[key]
qlib.init(provider_uri=str(root/spec['calendar_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2010-03-11',end_time='2026-10-09',freq='day'))
issuer,_=read_historical(root/spec['current_issuer_workbook'])
sh=issuer.reindex(calendar).shares_outstanding
assert sh.notna().all() and sh.gt(0).all()
provider=root/spec['derived_provider'];manifest_path=provider/'qlib_store_manifest.json'
if manifest_path.exists():
 manifest=json.loads(manifest_path.read_text())
 previous_path=card/'preflight_result.json'
 if previous_path.exists():
  prior=json.loads(previous_path.read_text());assert prior['plan_sha256']==sha(card/'plan.json')
 else:
  revision=json.loads((card/'budget_amendment_before_fit.json').read_text())
  prior=json.loads((card/'preflight_attempt_before_budget_amendment.json').read_text())
  assert prior['plan_sha256']==revision['old_plan_sha256'] and revision['new_plan_sha256']==sha(card/'plan.json')
 assert sha(manifest_path)==prior['derived_provider_manifest_sha256']
else:
 raw=pd.DataFrame({'instrument':'SOXX','datetime':calendar,'shares_outstanding':sh.to_numpy(float)})
 manifest=build_qlib_store_from_frame(raw,provider_uri=provider,freq='day',as_of='2026-10-21',source='iShares SOXX official fund download sha256='+spec['current_issuer_workbook_sha256'],field_map={'shares_outstanding':'shares_outstanding'})
qlib.init(provider_uri=str(provider),region='us',kernels=1)
queried=D.features(['SOXX'],['$shares_outstanding'],start_time='2010-03-11',end_time='2026-10-09',freq='day').xs('SOXX',level='instrument').sort_index()
provider_calendar=pd.DatetimeIndex(D.calendar(start_time='2010-03-11',end_time='2026-10-09',freq='day'))
assert provider_calendar.equals(calendar) and queried.index.equals(calendar)
assert np.allclose(queried['$shares_outstanding'].to_numpy(float),sh.to_numpy(float),rtol=1e-6,atol=1)
with R.uri_context(str(parent_card/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=parent['recorder_id'],experiment_id=parent['experiment_id'])
 original=rec.load_object('models/base_outer_71');base=rec.load_object('inputs/features')
 frame=rec.load_object('inputs/daily_frame');timing=rec.load_object('inputs/timing')
base_names=list(original.base_features);assert len(base_names)==39 and set(base_names).issubset(base.columns)
assert frame.index.equals(base.index) and frame.index.equals(timing.index)
shares=queried['$shares_outstanding'].astype(float)
flow=np.log(shares.shift(2)/shares.shift(7)).reindex(frame.index)
feature=flow.where(base.soxx_ret20.lt(0),0.0).rename(plan['single_feature']['name'])
forward=frame['soxx_forward_open_return'].astype(float)
parts=[forward.shift(-i) for i in range(5)]
valid=np.logical_and.reduce([np.isfinite(x.to_numpy(float))&(x.to_numpy(float)>-1) for x in parts])
label=pd.Series(np.log1p(pd.concat(parts,axis=1)).sum(axis=1).where(valid),index=frame.index,name='soxx_forward_five_open_log_return')
mature=pd.to_datetime(timing.return_available_at.shift(-4),utc=True)
cutoff=pd.Timestamp('2023-01-01',tz='America/New_York').tz_convert('UTC')
last=pd.Timestamp('2026-10-10',tz='America/New_York').tz_convert('UTC')
train_dates=(frame.index>=pd.Timestamp('2011-01-03'))&(frame.index<=pd.Timestamp('2022-12-30'))&(mature<cutoff)&label.notna()
held_dates=(frame.index>=pd.Timestamp('2023-01-03'))&(frame.index<=pd.Timestamp('2026-10-07'))&(mature<last)&label.notna()
assert feature.loc[train_dates].notna().all() and feature.loc[held_dates].notna().all()
assert train_dates.sum()>=plan['cohorts']['minimum_mature_train_rows'] and held_dates.sum()>=plan['cohorts']['minimum_mature_held_rows']
augmented=base[base_names].copy();augmented[feature.name]=feature
meta=pd.DataFrame({'label_available_at':mature,'train_eligible':train_dates,'held_eligible':held_dates},index=frame.index)
static=pd.concat({'feature':augmented,'label':label.to_frame(),'meta':meta},axis=1)
static.index=pd.MultiIndex.from_arrays([frame.index,['SOXL_SOXX_CASH']*len(frame)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train_signal']),'held':tuple(plan['cohorts']['held_reused_diagnostic'])})
train_prepared=dataset.prepare('train',col_set=['feature','label','meta'])
held_prepared=dataset.prepare('held',col_set=['feature','label','meta'])
assert len(train_prepared)>=train_dates.sum() and len(held_prepared)>=held_dates.sum()
report={'status':'conditional_source_and_DatasetH_maturity_preflight_passed_no_fit','plan_sha256':sha(card/'plan.json'),'source_raw_sha256':spec['current_issuer_workbook_sha256'],'source_vintage_audit_sha256':spec['workbook_vintage_audit_sha256'],'derived_provider_manifest_sha256':sha(manifest_path),'parent_recorder_id':parent['recorder_id'],'original_primary_features':len(base_names),'candidate_coordinates':1,'training_mature_rows':int(train_dates.sum()),'held_mature_rows':int(held_dates.sum()),'training_last_label_available_at':str(mature.loc[train_dates].max()),'held_last_label_available_at':str(mature.loc[held_dates].max()),'train_segment_rows':len(train_prepared),'held_segment_rows':len(held_prepared),'candidate_train_nonzero_rows':int(feature.loc[train_dates].ne(0).sum()),'candidate_held_nonzero_rows':int(feature.loc[held_dates].ne(0).sum()),'new_model_fits':0,'new_native_accounts':0,'held_scores_computed':False,'historical_independence':False,'historical_first_public_clock_qualified':False}
receipt=card/'preflight_result.json'
if receipt.exists():
 previous=json.loads(receipt.read_text());assert previous==report
else:atomic_json(report,receipt)
{k:report[k] for k in ('status','training_mature_rows','held_mature_rows','candidate_train_nonzero_rows','candidate_held_nonzero_rows')}


{'status': 'conditional_source_and_DatasetH_maturity_preflight_passed_no_fit',
 'training_mature_rows': 3014,
 'held_mature_rows': 940,
 'candidate_train_nonzero_rows': 1049,
 'candidate_held_nonzero_rows': 350}

## 执行门

先完成无训练 Qlib DatasetH 预检与独立设计复核；完整拟合代码和闸门在下一单元。

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from src.analysis.seed_disagreement_information import full_calendar_bootstrap
from src.data.nas_market_data import archive_nas_recorder, _ssh
import shlex
review_path=card/'design_review_final.json';permit_path=card/'execution_approval.json'
if not review_path.exists() or not permit_path.exists():
 print('preflight_only: independent design review and exact execution permit absent')
else:
 review=json.loads(review_path.read_text());permit=json.loads(permit_path.read_text())
 notebook=json.loads((card/'stage1_issuer_share_creation_information_20261022.ipynb').read_text())
 preflight_source=''.join(notebook['cells'][1]['source'])
 preflight_cell_hash=hashlib.sha256(preflight_source.encode()).hexdigest()
 fit_source=''.join(notebook['cells'][3]['source'])
 fit_hash=hashlib.sha256(fit_source.encode()).hexdigest()
 digest=sha(card/'plan.json');preflight_hash=sha(card/'preflight_result.json')
 assert review['verdict']=='APPROVE_DESIGN' and review['plan_sha256']==digest
 assert review['preflight_cell_source_sha256']==preflight_cell_hash
 assert review['fit_cell_source_sha256']==fit_hash
 assert permit['approved'] is True and permit['plan_sha256']==digest
 assert permit['review_sha256']==sha(review_path) and permit['fit_cell_source_sha256']==fit_hash
 assert permit['preflight_cell_source_sha256']==preflight_cell_hash
 assert permit['preflight_sha256']==preflight_hash
 assert report==json.loads((card/'preflight_result.json').read_text())
 state_path=card/'run_state.json';tracking=card/'qlib_recorder'
 state=json.loads(state_path.read_text()) if state_path.exists() else {'plan_sha256':digest,'fit_attempts':[],'fits_done':[],
   'artifact_sha256':{'models':{},'predictions':{}},'bootstrap_attempted':False,'new_native_accounts':0}
 assert state['plan_sha256']==digest
 if state.get('completed'):
  summary=json.loads((card/'summary.json').read_text())
  assert summary['plan_sha256']==digest and summary['nas_readback']['verified']
 elif state.get('analysis_done'):
  summary=json.loads((card/'summary.json').read_text())
 else:
  if state['bootstrap_attempted']:raise ValueError('blocked: fixed bootstrap attempt already consumed')
  tr=train_prepared.loc[train_prepared[('meta','train_eligible')].astype(bool)]
  he=held_prepared.loc[held_prepared[('meta','held_eligible')].astype(bool)]
  assert len(tr)==report['training_mature_rows'] and len(he)==report['held_mature_rows']
  yname='soxx_forward_five_open_log_return';name=plan['single_feature']['name']
  ytr=tr[('label',yname)].to_numpy(float);yhe=he[('label',yname)].to_numpy(float)
  assert np.isfinite(ytr).all() and np.isfinite(yhe).all()
  inputs={'baseline':(tr['feature'][base_names].to_numpy(float),he['feature'][base_names].to_numpy(float)),
          'candidate':(tr['feature'][base_names+[name]].to_numpy(float),he['feature'][base_names+[name]].to_numpy(float))}
  assert np.isfinite(inputs['candidate'][0][:,-1]).all() and np.isfinite(inputs['candidate'][1][:,-1]).all()
  models={};predictions={}
  start={'experiment_name':plan['experiment'],'uri':str(tracking)}
  if state.get('recorder_id'):start.update(recorder_id=state['recorder_id'],resume=True)
  with R.start(**start):
   recorder=R.get_recorder()
   if state.get('recorder_id'):
    assert state['recorder_id']==recorder.id and state['experiment_id']==str(recorder.experiment_id)
   state.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id))
   atomic_json(state,state_path)
   artifact_root=tracking/state['experiment_id']/state['recorder_id']/'artifacts'
   if not state.get('sources_saved'):
    recorder.save_objects(artifact_path='inputs',plan=plan,preflight=report,dataset=dataset,
                          source_audit=source_audit,workbook_vintage_audit=vintage)
    state['sources_saved']=True;atomic_json(state,state_path)
   for family in ('baseline','candidate'):
    xtrain,xheld=inputs[family]
    if family in state['fits_done']:
     assert sha(artifact_root/'models'/family)==state['artifact_sha256']['models'][family]
     model=recorder.load_object('models/'+family)
    else:
     if family in state['fit_attempts'] or len(state['fit_attempts'])>=plan['budget']['model_fit_attempts_max']:
      raise ValueError('blocked: fixed model fit attempt consumed')
     state['fit_attempts'].append(family);atomic_json(state,state_path)
     model=make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),Ridge(alpha=1.0))
     model.fit(xtrain,ytr)
     recorder.save_objects(artifact_path='models',**{family:model})
     state['artifact_sha256']['models'][family]=sha(artifact_root/'models'/family)
     state['fits_done'].append(family);atomic_json(state,state_path)
    if family in state.get('predictions_done',[]):
     assert sha(artifact_root/'predictions'/family)==state['artifact_sha256']['predictions'][family]
     saved=recorder.load_object('predictions/'+family)
     prediction=saved.prediction.to_numpy(float)
    else:
     if family in state.get('prediction_attempts',[]):raise ValueError('blocked: fixed prediction attempt consumed')
     state.setdefault('prediction_attempts',[]).append(family);atomic_json(state,state_path)
     prediction=model.predict(xheld)
     assert np.isfinite(prediction).all()
     saved=pd.DataFrame({'signal_date':he.index.get_level_values('datetime'),'prediction':prediction,'target':yhe})
     recorder.save_objects(artifact_path='predictions',**{family:saved})
     state['artifact_sha256']['predictions'][family]=sha(artifact_root/'predictions'/family)
     state.setdefault('predictions_done',[]).append(family);atomic_json(state,state_path)
    models[family],predictions[family]=model,prediction
    recorder.log_metrics(completed_fits=len(state['fits_done']),completed_predictions=len(state['predictions_done']))
   assert len(state['fits_done'])==2 and len(state['predictions_done'])==2
   dates=pd.DatetimeIndex(he.index.get_level_values('datetime'))
   daily=pd.DataFrame({'signal_date':dates,'target':yhe,'baseline_prediction':predictions['baseline'],
                       'candidate_prediction':predictions['candidate']})
   daily['baseline_squared_error']=(daily.target-daily.baseline_prediction)**2
   daily['candidate_squared_error']=(daily.target-daily.candidate_prediction)**2
   daily['paired_mse_improvement']=daily.baseline_squared_error-daily.candidate_squared_error
   assert daily.signal_date.is_monotonic_increasing and len(daily)==report['held_mature_rows']
   boundary=pd.Timestamp('2025-01-01')
   halves=[daily.loc[daily.signal_date.lt(boundary)],daily.loc[daily.signal_date.ge(boundary)]]
   assert min(map(len,halves))>300
   bootstrap=plan['information_gate']['paired_stationary_bootstrap']
   state['bootstrap_attempted']=True;atomic_json(state,state_path)
   interval,_,replicates=full_calendar_bootstrap(daily.paired_mse_improvement.to_numpy(float),
     mean_block=bootstrap['mean_block_sessions'],resamples=bootstrap['draws'],seed=bootstrap['seed'],
     alpha=bootstrap['one_sided_alpha'])
   coefficient=float(models['candidate'].named_steps['ridge'].coef_[-1])
   metrics={'baseline_held_mse':float(daily.baseline_squared_error.mean()),
            'candidate_held_mse':float(daily.candidate_squared_error.mean()),
            'held_paired_mse_improvement':float(daily.paired_mse_improvement.mean()),
            'held_half_paired_improvements':[float(x.paired_mse_improvement.mean()) for x in halves],
            'candidate_standardized_coordinate_coefficient':coefficient,
            'bootstrap_lower':interval['lower_bound'],'bootstrap_upper':interval['upper_bound'],
            'bootstrap':interval}
   gates={'coefficient_positive':coefficient>0,
          'full_held_mean_positive':metrics['held_paired_mse_improvement']>0,
          'both_halves_positive':all(x>0 for x in metrics['held_half_paired_improvements']),
          'bootstrap_lower_positive':metrics['bootstrap_lower']>0}
   passed=bool(all(gates.values()))
   daily['quarter']=daily.signal_date.dt.to_period('Q').astype(str)
   quarters=[{'quarter':q,'sessions':len(group),'paired_mean_mse_improvement':float(group.paired_mse_improvement.mean())}
             for q,group in daily.groupby('quarter',sort=True)]
   recorder.save_objects(artifact_path='analysis/daily',held=daily)
   recorder.save_objects(artifact_path='analysis',metrics=metrics,gates=gates,quarters=quarters,
                         bootstrap_replicates=replicates,information_gate_pass=passed)
   recorder.log_metrics(held_paired_mse_improvement=metrics['held_paired_mse_improvement'],
                        bootstrap_lower=metrics['bootstrap_lower'],information_gate_pass=int(passed))
  summary={'experiment':plan['experiment'],'plan_sha256':digest,
    'decision':'conditional_development_information_pass_source_PIT_still_required' if passed else
               'not_promoted_fixed_issuer_share_creation_information_failed',
    'objective_achieved':False,'information_gate_pass':passed,
    'training_mature_rows':len(tr),'held_mature_rows':len(he),'metrics':metrics,'gates':gates,
    'quarterly_information_attribution':quarters,'new_model_fits':2,'new_bootstraps':1,'new_native_accounts':0,
    'reused_market_history':True,'historical_first_public_clock_qualified':False,
    'independent_forward_qualified_days':0,'recorder_id':state['recorder_id'],
    'experiment_id':state['experiment_id'],'recorder_uri':str(tracking),
    'recorder_local_dir':str(tracking/state['experiment_id']/state['recorder_id']),
    'derived_provider_manifest_sha256':report['derived_provider_manifest_sha256'],
    'artifact_policy':plan['artifact_policy'],
    'limitations':['Current issuer workbook is retrospective; two-Qlib-session delay is an as-if timing assumption',
                   'Exact 2021/22/23 vintage shares do not certify first publication on every historical date',
                   'Information MSE is not a fee-net account result, nor independent forward evidence']}
  atomic_json(summary,card/'summary.json')
  atomic_json({'status':summary['decision'],'plan_sha256':digest,'recorder_id':state['recorder_id'],
    'metrics':metrics,'gates':gates,'new_model_fits':2,'new_native_accounts':0,
    'historical_first_public_clock_qualified':False},card/'result_review.json')
  pd.DataFrame(quarters).to_csv(card/'quarterly_information_attribution.csv',index=False)
  artifacts=tracking/state['experiment_id']/state['recorder_id']/'artifacts'
  atomic_json({'recorder_id':state['recorder_id'],
    'entries':[{'path':str(x.relative_to(card)),'sha256':sha(x),'bytes':x.stat().st_size}
               for x in sorted(artifacts.rglob('*')) if x.is_file()]},card/'verification_manifest.json')
  state['analysis_done']=True;atomic_json(state,state_path)
 if state.get('analysis_done') and not state.get('completed'):
  from src.data.nas_market_data import _ssh
  manifest=json.loads((card/'verification_manifest.json').read_text())
  assert manifest['recorder_id']==summary['recorder_id']
  for entry in manifest['entries']:
   path=card/entry['path'];assert sha(path)==entry['sha256'] and path.stat().st_size==entry['bytes']
  binding_path=card/'nas_binding.json'
  if not binding_path.exists():
   binding=json.loads((root/'research/stage1_rolling126_shortfall_objective_20261022/nas_binding.json').read_text())
   assert binding.get('verified') is True and binding.get('authority')=='yanxuan_nas'
   binding.pop('evidence',None);binding['research_card']=str(card.relative_to(root));atomic_json(binding,binding_path)
  if 'nas_evidence' not in summary:
   summary['nas_evidence']=archive_nas_recorder(binding_path,workspace_root=root,card=card,
      recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id'])
  binding=json.loads(binding_path.read_text());evidence=summary['nas_evidence']
  assert binding['evidence']['sha256']==evidence['sha256']
  remote_sha=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]
  assert remote_sha==evidence['sha256']
  summary['nas_readback']={'sha256':remote_sha,'bytes':evidence['size_bytes'],'verified':True}
  atomic_json(summary,card/'summary.json')
  state['completed']=True;atomic_json(state,state_path)
 print({k:summary[k] for k in ('decision','information_gate_pass','new_model_fits','new_native_accounts','recorder_id')})


## 中文结论

预注册条件性开发期信息检验。执行前无模型拟合或账户；最终结果以summary.json、result_review.json和原生Qlib Recorder为准。通过信息门也不认证历史PIT或允许0bp账户。